In [ ]:
SELECT
    srv.description  AS service_description,
    atyb.description AS activity_type_description,
    COUNT(*) AS row_count
FROM silver_wip_activityentry a
LEFT JOIN silver_wip_service srv
    ON a.activity_service_id = srv.id
LEFT JOIN silver_wip_activitytype atyb
    ON a.activity_type_id = atyb.id
WHERE LOWER(COALESCE(srv.description, '')) LIKE '%cancel%'
   OR LOWER(COALESCE(atyb.description, '')) LIKE '%cancel%'
GROUP BY
    srv.description,
    atyb.description
ORDER BY row_count DESC;

In [ ]:
SELECT
    CASE
        WHEN a.is_dna = true THEN 'Did Not Attend'
        WHEN LOWER(COALESCE(srv.description, atyb.description, '')) LIKE '%cancel%' 
            THEN 'Cancelled with greater than 24 hours notice'
        WHEN a.activity_date_time > current_timestamp() THEN 'Booked'
        WHEN a.activity_date_time < current_timestamp() THEN 'Attended'
        ELSE 'Unknown'
    END AS expected_session_status_src_name,
    COUNT(*) AS row_count
FROM silver_wip_activityentry a
LEFT JOIN silver_wip_service srv
    ON a.activity_service_id = srv.id
LEFT JOIN silver_wip_activitytype atyb
    ON a.activity_type_id = atyb.id
GROUP BY
    CASE
        WHEN a.is_dna = true THEN 'Did Not Attend'
        WHEN LOWER(COALESCE(srv.description, atyb.description, '')) LIKE '%cancel%' 
            THEN 'Cancelled with greater than 24 hours notice'
        WHEN a.activity_date_time > current_timestamp() THEN 'Booked'
        WHEN a.activity_date_time < current_timestamp() THEN 'Attended'
        ELSE 'Unknown'
    END
ORDER BY row_count DESC;